In [1]:
!nvidia-smi

Tue Apr 21 18:59:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/AhmedAbdElkhalek16/acute-triage-system.git
%cd acute-triage-system
!pip install -q timm albumentations grad-cam gradio
print("✅ Ready!")

Cloning into 'acute-triage-system'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 23 (delta 1), reused 20 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 117.69 KiB | 1.87 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/acute-triage-system
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 75.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Ready!


In [3]:
from google.colab import files
import os

os.makedirs('models/weights', exist_ok=True)
print("ارفع ملف xray_best.pth")
uploaded = files.upload()

!mv xray_best.pth models/weights/xray_best.pth
print("✅ Model ready!")

ارفع ملف xray_best.pth


Saving xray_best.pth to xray_best.pth
✅ Model ready!


In [4]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.0 MB/s eta 0:00:00


In [5]:
import sys
sys.path.insert(0, '/content/acute-triage-system')

import torch
import numpy as np
import cv2
import gradio as gr
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from Src.preprocessing import get_val_transforms
from Src.models import get_model

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
THRESHOLD = 0.7
CLASSES   = ['Normal', 'Pneumonia']

model = get_model('xray', num_classes=2, pretrained=False, device=DEVICE)
model.load_state_dict(torch.load('models/weights/xray_best.pth',
                                  map_location=DEVICE))
model.eval()

target_layer = [model.backbone.conv_head]
transforms   = get_val_transforms(512)

print(f"✅ Model loaded on {DEVICE}")

✅ Model loaded on cuda


In [6]:
def predict(image):
    if image is None:
        return None, "❌ No image uploaded"

    # 1. Preprocess
    img_rgb = image
    tensor  = transforms(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)

    # 2. Predict
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pneumonia_prob = float(probs[1])
    normal_prob    = float(probs[0])
    predicted      = 1 if pneumonia_prob >= THRESHOLD else 0

    # 3. Grad-CAM
    cam           = GradCAM(model=model, target_layers=target_layer)
    targets       = [ClassifierOutputTarget(predicted)]
    grayscale_cam = cam(input_tensor=tensor, targets=targets)[0]

    img_float = cv2.resize(img_rgb, (512, 512)).astype(np.float32) / 255.0
    cam_image = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

    # 4. Priority
    if predicted == 1:
        if pneumonia_prob >= 0.90:
            priority = "🔴 CRITICAL — فوري جداً (أقل من 15 دقيقة)"
        elif pneumonia_prob >= 0.70:
            priority = "🟠 HIGH — خلال ساعة"
        else:
            priority = "🟡 MEDIUM — خلال 4 ساعات"
    else:
        priority = "🟢 LOW — روتيني"

    # 5. Report
    report = f"""
## نتيجة التشخيص

| | |
|---|---|
| **التشخيص** | {'🫁 Pneumonia — التهاب رئوي' if predicted == 1 else '✅ Normal — سليم'} |
| **الأولوية** | {priority} |
| **Pneumonia** | {pneumonia_prob*100:.1f}% |
| **Normal** | {normal_prob*100:.1f}% |

> ⚕ للأغراض البحثية فقط — ليس للاستخدام الطبي
    """

    return cam_image, report

print("✅ Predict function ready!")

✅ Predict function ready!


In [7]:
with gr.Blocks(title="Acute Triage System", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🏥 Acute Findings Triage System")
    gr.Markdown("ارفع صورة أشعة صدر — النظام هيحدد وجود التهاب رئوي ومستوى الأولوية")

    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(
                label="📤 ارفع صورة الأشعة",
                type="numpy",
                height=380
            )
            run_btn = gr.Button("🔍 تشخيص", variant="primary", size="lg")

        with gr.Column(scale=1):
            cam_output = gr.Image(
                label="🧠 Grad-CAM — المناطق اللي النموذج بص عليها",
                height=380
            )

    report_output = gr.Markdown()

    run_btn.click(
        fn=predict,
        inputs=[img_input],
        outputs=[cam_output, report_output]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_579/1016653154.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Acute Triage System", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fcf08aa6ae01fb0117.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fcf08aa6ae01fb0117.gradio.live
